## Phase 1: Load and split

In [2]:
import numpy as np
import pandas as pd

ratings = pd.read_csv("../data/ml-25m/ratings.csv",
                      dtype={"userId": "int32", "movieId": "int32",
                             "rating": "float32", "timestamp": "int64"})
movies = pd.read_csv("../data/ml-25m/movies.csv")

ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")

print(ratings.shape, movies.shape)
print("date range:", ratings.date.min().date(), "to", ratings.date.max().date())
movies.head()

(25000095, 5) (62423, 3)
date range: 1995-01-09 to 2019-11-21


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings.head()

,userId,movieId,rating,timestamp,date
0,1,296,5.0,1147880044,2006-05-17 15:34:04
1,1,306,3.5,1147868817,2006-05-17 12:26:57
2,1,307,5.0,1147868828,2006-05-17 12:27:08
3,1,665,5.0,1147878820,2006-05-17 15:13:40
4,1,899,3.5,1147868510,2006-05-17 12:21:50


In [4]:
# A recommender needs enough signal per user and per item to learn from.
# Keep users with >= 20 ratings and movies with >= 50.
u_counts = ratings.userId.value_counts()
m_counts = ratings.movieId.value_counts()

r = ratings[ratings.userId.isin(u_counts[u_counts >= 20].index) & ratings.movieId.isin(m_counts[m_counts >= 50].index)]

print(f"kept {len(r):,} of {len(ratings):,} ratings")
print(f"{r.userId.nunique():,} users, {r.movieId.nunique():,} movies")

kept 24,644,928 of 25,000,095 ratings
162,540 users, 13,176 movies


In [5]:
r.head()

,userId,movieId,rating,timestamp,date
0,1,296,5.0,1147880044,2006-05-17 15:34:04
1,1,306,3.5,1147868817,2006-05-17 12:26:57
2,1,307,5.0,1147868828,2006-05-17 12:27:08
3,1,665,5.0,1147878820,2006-05-17 15:13:40
4,1,899,3.5,1147868510,2006-05-17 12:21:50


In [6]:
u_counts.head()

userId
72315     32202
80974      9178
137293     8913
33844      7919
20055      7488
Name: count, dtype: int64

In [7]:
m_counts.head()

movieId
356     81491
318     81482
296     79672
593     74127
2571    72674
Name: count, dtype: int64

In [8]:
# Time-based split: train on everything before the cutoff, test on what comes after.
CUTOFF = "2018-01-01"

train = r[r.date <  CUTOFF]
test  = r[r.date >= CUTOFF]

# Only evaluate on users we've actually seen before, and items that exist in train.
known_users = set(train.userId.unique())
known_items = set(train.movieId.unique())
test = test[test.userId.isin(known_users) & test.movieId.isin(known_items)]

print(f"train: {len(train):,} ratings, {train.userId.nunique():,} users")
print(f"test : {len(test):,} ratings, {test.userId.nunique():,} users")

train: 22,261,221 ratings, 150,543 users
test : 414,622 ratings, 6,460 users


In [9]:
# A "relevant" item is one the user rated highly. That's what we're trying to surface.
LIKE_THRESHOLD = 4.0

test_liked = (test[test.rating >= LIKE_THRESHOLD]
              .groupby("userId")["movieId"].apply(set))

# Users need at least one liked item in the test window to be evaluable
test_liked = test_liked[test_liked.apply(len) >= 1]

print(f"{len(test_liked):,} evaluable users")
print(f"median liked items per user: {test_liked.apply(len).median():.0f}")

6,269 evaluable users
median liked items per user: 13


In [10]:
test_liked.sample(10)

userId
107251    {122916, 122918, 170697, 168492, 178061, 11566...
19845     {1732, 296, 71464, 128360, 7307, 4973, 4878, 6...
132649                     {170705, 171011, 176601, 163134}
18540     {122916, 122918, 122920, 88140, 86190, 164179,...
61810     {69122, 1288, 122890, 122892, 111759, 93840, 8...
104522    {1672, 5420, 2989, 95567, 3984, 7573, 4027, 3068}
140861                                       {7669, 107406}
81751     {170945, 1732, 177765, 106022, 168250, 152081,...
143832    {2951, 8970, 3083, 6539, 8977, 98961, 8724, 44...
78663     {166528, 122916, 34405, 122918, 174055, 122926...
Name: movieId, dtype: object

---
## Phase 2: The evaluation harness

### The metrics

In [13]:
def recall_at_k(recommended, relevant, k):
    """Of everything the user actually liked, what fraction did we surface in the top k?"""
    if not relevant:
        return np.nan
    return len(set(recommended[:k]) & relevant) / len(relevant)

def precision_at_k(recommended, relevant, k):
    """Of the k items we recommended, what fraction did the user actually like?"""
    return len(set(recommended[:k]) & relevant) / k

def ndcg_at_k(recommended, relevant, k):
    """Like recall, but rewards putting hits near the top of the list."""
    if not relevant:
        return np.nan
    # Discount each position by log2(rank + 1): position 1 counts full, lower ranks less
    dcg = sum((1.0 if item in relevant else 0.0) / np.log2(i + 2)
              for i, item in enumerate(recommended[:k]))
    # Best possible: every hit packed into the top positions
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else np.nan

### The evaluation loop

In [15]:
K = 10

def evaluate(recommend_fn, test_liked, k=K, n_users=None, seed=42):
    """Score a recommender across users.

    recommend_fn(user_id, k) -> list of movieIds, best first
    """
    users = test_liked.index
    if n_users is not None and n_users < len(users):
        users = pd.Series(users).sample(n_users, random_state=seed).values

    rows = []
    for u in users:
        relevant = test_liked.loc[u]
        recs = recommend_fn(u, k)
        rows.append({
            "recall":    recall_at_k(recs, relevant, k),
            "precision": precision_at_k(recs, relevant, k),
            "ndcg":      ndcg_at_k(recs, relevant, k),
        })

    return pd.DataFrame(rows).mean().round(4)

### Sanity-check the harness before trusting it

In [17]:
all_items = train.movieId.unique()
rng = np.random.default_rng(42)

def random_rec(user_id, k):
    return list(rng.choice(all_items, size=k, replace=False))

print("random recommender:")
print(evaluate(random_rec, test_liked, n_users=500).to_string())

random recommender:
recall       0.0020
precision    0.0026
ndcg         0.0028
